# FNCE 449 — Day 1 (Part A): Financial Data Management

In this notebook we build a stock price database from scratch and learn the core `pandas` workflow that underpins almost every empirical finance project:

| Section | Topic |
|---|---|
| 1 | Building a `DataFrame` |
| 2 | Loading market data from an API |
| 3 | Inspecting the data |
| 4 | Selecting and slicing |
| 5 | Combining datasets |
| 6 | Summary statistics and missing values |
| 7 | Creating new variables |
| 8 | Grouping and returns |
| 9 | Reshaping: pivots and multi-level indices |
| 10 | OLS regression |

## 1. Building a `DataFrame`

A **library** is a collection of ready-made functions and objects, so you don't have to write everything yourself.

**`pandas`** is the standard Python library for data manipulation and analysis. Its central object is the `DataFrame` — think of it as a spreadsheet that you drive with code instead of with a mouse.

We will build a database of daily prices and volumes for five stocks:

| Ticker | Company |
|---|---|
| AAPL | Apple Inc. |
| AMZN | Amazon.com Inc. |
| NVDA | NVIDIA Corporation |
| CSCO | Cisco Systems Inc. |
| CGC | Canopy Growth Corp. |

In [11]:
# If pandas is not installed in this environment, install it first.

import pandas as pd

Start small: a `DataFrame` with a single column, **symbol**, holding the five tickers.

In [12]:
metadata = pd.DataFrame(['AAPL', 'AMZN', 'NVDA', 'CSCO', 'CGC'], columns=["symbol"])
metadata

,symbol
0,AAPL
1,AMZN
2,NVDA
3,CSCO
4,CGC


Two things to notice in the output: the column name **symbol** is bolded on top of the data, and there is a default *index* column on the left. Like all Python indices, it starts at zero.

Adding a second column is a one-liner — we just assign to a column name that does not exist yet:

In [13]:
metadata["sector"] = ("Technology", "Consumer Discretionary", "Technology",
                      "Telecommunications", "Healthcare")
metadata

,symbol,sector
0,AAPL,Technology
1,AMZN,Consumer Discretionary
2,NVDA,Technology
3,CSCO,Telecommunications
4,CGC,Healthcare


## 2. Loading market data from an API

Real price data comes from a data vendor. We use **Databento**, a professional-grade market data provider.

An **API** (Application Programming Interface) is the mechanism that lets one program request a resource from another. The program asking is the *client* (us); the program holding the data is the *server* (Databento). The API defines the vocabulary we use to state what we want, so the server can understand and fulfil the request. Access requires an **API key** — effectively a password tied to our account.

The function below wraps a single Databento endpoint: it sends our request, receives a CSV, and hands back a `DataFrame`.

In [ ]:
def get_range(api_key, dataset, symbols, schema, start, end, limit=None):
    """Query the Databento historical API and return the result as a DataFrame."""
    import base64
    import sys
    import types
    import urllib.parse
    import urllib.request

    if "requests" not in sys.modules:
        requests_module = types.ModuleType("requests")
        auth_module = types.ModuleType("requests.auth")

        class HTTPBasicAuth:
            def __init__(self, username, password):
                self.username = username
                self.password = password

        def post(url, auth=None, data=None):
            headers = {}
            if auth is not None:
                token = base64.b64encode(
                    f"{auth.username}:{auth.password}".encode("utf-8")
                ).decode("ascii")
                headers["Authorization"] = f"Basic {token}"

            body = urllib.parse.urlencode(data or {}).encode("utf-8")
            req = urllib.request.Request(url, data=body, headers=headers, method="POST")
            with urllib.request.urlopen(req) as resp:
                content = resp.read()

            return types.SimpleNamespace(content=content)

        auth_module.HTTPBasicAuth = HTTPBasicAuth
        requests_module.post = post
        requests_module.auth = auth_module
        sys.modules["requests"] = requests_module
        sys.modules["requests.auth"] = auth_module
    from io import StringIO
    from requests.auth import HTTPBasicAuth

    url = 'https://hist.databento.com/v0/timeseries.get_range'
    auth = HTTPBasicAuth(api_key, '')

    payload = {
        'dataset': dataset,
        'symbols': symbols,
        'schema': schema,
        'start': start,
        'end': end,
        'encoding': 'csv',
        'pretty_px': 'true',
        'pretty_ts': 'true',
        'map_symbols': 'true',
        'limit': limit,
    }

    response = requests.post(url, auth=auth, data=payload)
    return pd.read_csv(StringIO(response.content.decode('utf-8')))

Now we set the parameters of the request. The `ohlcv-1d` schema gives us one row per stock per day, with the **o**pen, **h**igh, **l**ow, **c**lose prices and the **v**olume.

In [15]:
api_key = "db-Xx67tauyVWV8nYQktfgfApAWdTTkC"
dataset = "XNAS.ITCH"
list_symbols = ['AAPL', 'AMZN', 'NVDA', 'CSCO', 'CGC']
symbols = ','.join(list_symbols)
schema = 'ohlcv-1d'
start = '2020-01-01'
end = '2025-04-01'

In [16]:
StockPrices = get_range(api_key, dataset, symbols, schema, start, end)
StockPrices['ts_event'] = pd.to_datetime(StockPrices['ts_event'])

ModuleNotFoundError: No module named 'requests'

## 3. Inspecting the data

Before doing anything with a dataset, look at it. `head()` shows the first rows, `tail()` the last, `len()` the number of rows, and `.columns` the variable names.

In [7]:
StockPrices.head()

,ts_event,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
0,2020-01-02 00:00:00+00:00,35,2,13,295.25,300.93,295.080,300.85,8242956,AAPL
1,2020-01-02 00:00:00+00:00,35,2,393,1861.00,1900.26,1861.000,1899.50,1123936,AMZN
2,2020-01-02 00:00:00+00:00,35,2,1727,48.17,48.50,47.880,48.50,4340490,CSCO
3,2020-01-02 00:00:00+00:00,35,2,5704,237.55,240.10,236.750,240.01,1408689,NVDA
4,2020-01-02 00:00:00+00:00,35,2,1361,21.30,21.76,19.885,20.08,697608,CGC


In [8]:
StockPrices.tail()

,ts_event,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
6585,2025-03-31 00:00:00+00:00,35,2,2999,0.985,0.985,0.881,0.906,1192827,CGC
6586,2025-03-31 00:00:00+00:00,35,2,38,216.240,224.150,214.910,221.130,26807329,AAPL
6587,2025-03-31 00:00:00+00:00,35,2,3745,60.600,61.890,60.240,61.510,12446080,CSCO
6588,2025-03-31 00:00:00+00:00,35,2,853,190.160,191.350,184.640,189.180,20788937,AMZN
6589,2025-03-31 00:00:00+00:00,35,2,11667,106.850,108.960,103.660,107.520,73624332,NVDA


In [9]:
len(StockPrices), StockPrices.columns.to_list()

(6590,
 ['ts_event',
  'rtype',
  'publisher_id',
  'instrument_id',
  'open',
  'high',
  'low',
  'close',
  'volume',
  'symbol'])

`iloc` selects rows by *position* (as opposed to `loc`, which selects by index label — more on that below):

In [10]:
StockPrices.iloc[10:20]

,ts_event,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
10,2020-01-06 00:00:00+00:00,35,2,13,295.53,299.96,292.75,299.01,6390771,AAPL
11,2020-01-06 00:00:00+00:00,35,2,393,1857.00,1903.33,1852.50,1899.95,1129524,AMZN
12,2020-01-06 00:00:00+00:00,35,2,5707,231.90,237.37,230.80,237.00,1930926,NVDA
13,2020-01-06 00:00:00+00:00,35,2,1726,47.24,47.85,47.02,47.70,5098839,CSCO
14,2020-01-06 00:00:00+00:00,35,2,1360,19.93,20.25,19.48,20.14,374922,CGC
15,2020-01-07 00:00:00+00:00,35,2,1725,47.83,47.83,46.55,46.80,4566256,CSCO
16,2020-01-07 00:00:00+00:00,35,2,13,300.40,300.95,293.75,294.51,5968500,AAPL
17,2020-01-07 00:00:00+00:00,35,2,1360,20.25,20.42,19.50,19.50,296221,CGC
18,2020-01-07 00:00:00+00:00,35,2,5704,238.20,241.76,235.01,235.50,1805231,NVDA
19,2020-01-07 00:00:00+00:00,35,2,393,1906.01,1913.90,1872.84,1876.00,995475,AMZN


How many — and which — distinct tickers are in the data? Select the `symbol` column, then ask for its unique values. The syntax is nested: `.unique()` returns an array, and `.tolist()` converts it to a Python list.

In [11]:
StockPrices['symbol'].unique().tolist()

['AAPL', 'AMZN', 'CSCO', 'NVDA', 'CGC']

## 4. Selecting and slicing

### Selecting by index

Setting a meaningful column as the *index* makes retrieval much easier. Here the natural choice is the date:

In [12]:
StockPrices = StockPrices.set_index('ts_event')
StockPrices.head()

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2020-01-02 00:00:00+00:00,35,2,13,295.25,300.93,295.080,300.85,8242956,AAPL
2020-01-02 00:00:00+00:00,35,2,393,1861.00,1900.26,1861.000,1899.50,1123936,AMZN
2020-01-02 00:00:00+00:00,35,2,1727,48.17,48.50,47.880,48.50,4340490,CSCO
2020-01-02 00:00:00+00:00,35,2,5704,237.55,240.10,236.750,240.01,1408689,NVDA
2020-01-02 00:00:00+00:00,35,2,1361,21.30,21.76,19.885,20.08,697608,CGC


With the date as index, pulling every price on a given day takes one call to `loc`:

In [13]:
StockPrices.loc['2021-03-15']

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2021-03-15 00:00:00+00:00,35,2,1662,34.21,36.11,34.17,35.90,913738,CGC
2021-03-15 00:00:00+00:00,35,2,479,3076.29,3100.00,3032.81,3073.00,867961,AMZN
2021-03-15 00:00:00+00:00,35,2,2089,48.71,49.55,48.64,49.41,5287101,CSCO
2021-03-15 00:00:00+00:00,35,2,21,120.98,124.55,120.42,124.10,21268632,AAPL
2021-03-15 00:00:00+00:00,35,2,6560,514.24,529.09,510.20,527.80,1713427,NVDA


### Slicing by condition

We can also keep only the rows that satisfy a logical condition. To isolate Cisco, we impose `StockPrices['symbol'] == "CSCO"`:

In [14]:
CSCO = StockPrices[StockPrices['symbol'] == "CSCO"]
CSCO.head()

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2020-01-02 00:00:00+00:00,35,2,1727,48.17,48.50,47.880,48.50,4340490,CSCO
2020-01-03 00:00:00+00:00,35,2,1727,47.96,48.13,47.480,47.60,4991259,CSCO
2020-01-06 00:00:00+00:00,35,2,1726,47.24,47.85,47.020,47.70,5098839,CSCO
2020-01-07 00:00:00+00:00,35,2,1725,47.83,47.83,46.550,46.80,4566256,CSCO
2020-01-08 00:00:00+00:00,35,2,1725,47.38,47.79,46.815,47.79,6597831,CSCO


Conditions combine with `&` (and) and `|` (or). Each condition needs its own parentheses:

In [15]:
AAPL = StockPrices[StockPrices['symbol'] == "AAPL"]
CSCO_AAPL = StockPrices[(StockPrices['symbol'] == "CSCO") |
                        (StockPrices['symbol'] == "AAPL")]

# always check the slice did what you expected
CSCO_AAPL['symbol'].unique().tolist()

['AAPL', 'CSCO']

## 5. Combining datasets

There are two distinct ways to put datasets together.

**`pd.concat()` stacks rows** — use it when two `DataFrame`s hold the same variables for different observations:

In [16]:
TechStocks = pd.concat([AAPL, CSCO])
TechStocks.head()

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2020-01-02 00:00:00+00:00,35,2,13,295.25,300.93,295.08,300.85,8242956,AAPL
2020-01-03 00:00:00+00:00,35,2,13,297.20,300.58,294.31,297.79,7340919,AAPL
2020-01-06 00:00:00+00:00,35,2,13,295.53,299.96,292.75,299.01,6390771,AAPL
2020-01-07 00:00:00+00:00,35,2,13,300.40,300.95,293.75,294.51,5968500,AAPL
2020-01-08 00:00:00+00:00,35,2,13,296.51,304.43,296.51,303.58,8224184,AAPL


**`merge()` joins columns** — use it when two `DataFrame`s hold different variables about the same entities. Recall `metadata`, which records each stock's sector. We attach it to `StockPrices`, matching on `symbol`:

- `on="symbol"` is the *key* the two tables share;
- `how="left"` keeps every row of `StockPrices`, whether or not a match is found.

In [17]:
StockPrices = StockPrices.reset_index().merge(metadata, on="symbol", how="left")
StockPrices.tail()

,ts_event,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol,sector
6585,2025-03-31 00:00:00+00:00,35,2,2999,0.985,0.985,0.881,0.906,1192827,CGC,Healthcare
6586,2025-03-31 00:00:00+00:00,35,2,38,216.240,224.150,214.910,221.130,26807329,AAPL,Technology
6587,2025-03-31 00:00:00+00:00,35,2,3745,60.600,61.890,60.240,61.510,12446080,CSCO,Telecommunications
6588,2025-03-31 00:00:00+00:00,35,2,853,190.160,191.350,184.640,189.180,20788937,AMZN,Consumer Discretionary
6589,2025-03-31 00:00:00+00:00,35,2,11667,106.850,108.960,103.660,107.520,73624332,NVDA,Technology


Finally, we reorder the columns into something readable by passing a list of names:

In [18]:
lcolms = ['ts_event', 'symbol', 'sector', 'open', 'high', 'low', 'close', 'volume']
StockPrices = StockPrices[lcolms]
StockPrices.head()

,ts_event,symbol,sector,open,high,low,close,volume
0,2020-01-02 00:00:00+00:00,AAPL,Technology,295.25,300.93,295.080,300.85,8242956
1,2020-01-02 00:00:00+00:00,AMZN,Consumer Discretionary,1861.00,1900.26,1861.000,1899.50,1123936
2,2020-01-02 00:00:00+00:00,CSCO,Telecommunications,48.17,48.50,47.880,48.50,4340490
3,2020-01-02 00:00:00+00:00,NVDA,Technology,237.55,240.10,236.750,240.01,1408689
4,2020-01-02 00:00:00+00:00,CGC,Healthcare,21.30,21.76,19.885,20.08,697608


## 6. Summary statistics and missing values

`describe()` reports the standard summary statistics for every numeric column at once.

In [19]:
for column in ['open', 'high', 'low', 'close', 'volume']:
    StockPrices[column] = StockPrices[column].map(float)

StockPrices.describe()

,open,high,low,close,volume
count,6590.000000,6590.000000,6590.000000,6590.000000,6.590000e+03
mean,414.058315,420.472375,407.329268,414.017159,1.013752e+07
std,853.794151,865.523497,840.936000,853.054936,1.305551e+07
min,0.369900,0.397000,0.346000,0.373800,9.551500e+04
25%,45.832500,46.405000,45.242500,45.840000,1.668969e+06
50%,133.555000,135.740000,131.460000,133.785000,7.044324e+06
75%,232.887500,237.150000,229.230000,232.845000,1.304866e+07
max,3742.000000,3773.020000,3696.440000,3722.000000,2.134455e+08


Individual statistics are available as methods on a single column:

In [20]:
StockPrices['close'].mean(), StockPrices['close'].std(), StockPrices['close'].median()

(np.float64(414.0171589377845),
 np.float64(853.0549359750731),
 np.float64(133.78500000000003))

These numbers are close to meaningless, though: they pool prices across five very different stocks. Any statistic worth reporting has to be computed *within* a stock — so we slice first, then aggregate:

In [21]:
StockPrices[StockPrices['symbol'] == "AAPL"]['close'].std()

np.float64(67.67670306298926)

### Missing values

Missing observations can be filled forward (`ffill`, carry the last valid value forward), backward (`bfill`, use the next valid value), or with a constant (`fillna`).

These methods **return a new `DataFrame` rather than modifying the original**, so you have to assign the result to keep it:

In [22]:
StockPrices = StockPrices.ffill()   # carry the last observation forward
StockPrices.head()

,ts_event,symbol,sector,open,high,low,close,volume
0,2020-01-02 00:00:00+00:00,AAPL,Technology,295.25,300.93,295.080,300.85,8242956.0
1,2020-01-02 00:00:00+00:00,AMZN,Consumer Discretionary,1861.00,1900.26,1861.000,1899.50,1123936.0
2,2020-01-02 00:00:00+00:00,CSCO,Telecommunications,48.17,48.50,47.880,48.50,4340490.0
3,2020-01-02 00:00:00+00:00,NVDA,Technology,237.55,240.10,236.750,240.01,1408689.0
4,2020-01-02 00:00:00+00:00,CGC,Healthcare,21.30,21.76,19.885,20.08,697608.0


## 7. Creating new variables

Viewing and slicing is only half the job — we also want to **build** new variables out of existing columns.

Start with the ratio of the day's high to its low. It is a simple measure of intraday volatility: how much the price moved within the day.

In [23]:
StockPrices['RangeVolatility'] = StockPrices['high'] / StockPrices['low']
StockPrices.head()

,ts_event,symbol,sector,open,high,low,close,volume,RangeVolatility
0,2020-01-02 00:00:00+00:00,AAPL,Technology,295.25,300.93,295.080,300.85,8242956.0,1.019825
1,2020-01-02 00:00:00+00:00,AMZN,Consumer Discretionary,1861.00,1900.26,1861.000,1899.50,1123936.0,1.021096
2,2020-01-02 00:00:00+00:00,CSCO,Telecommunications,48.17,48.50,47.880,48.50,4340490.0,1.012949
3,2020-01-02 00:00:00+00:00,NVDA,Technology,237.55,240.10,236.750,240.01,1408689.0,1.014150
4,2020-01-02 00:00:00+00:00,CGC,Healthcare,21.30,21.76,19.885,20.08,697608.0,1.094292


### `.map()` — apply an existing function elementwise

Because $\log(1+x) \approx x$ for small $x$, taking logs of the ratio turns it into something that reads like a net return. `np.log` is a ready-made function, so we can hand it straight to `.map()`:

In [24]:
import numpy as np

StockPrices['LogVolatility'] = StockPrices['RangeVolatility'].map(np.log)
StockPrices.head()

,ts_event,symbol,sector,open,high,low,close,volume,RangeVolatility,LogVolatility
0,2020-01-02 00:00:00+00:00,AAPL,Technology,295.25,300.93,295.080,300.85,8242956.0,1.019825,0.019631
1,2020-01-02 00:00:00+00:00,AMZN,Consumer Discretionary,1861.00,1900.26,1861.000,1899.50,1123936.0,1.021096,0.020877
2,2020-01-02 00:00:00+00:00,CSCO,Telecommunications,48.17,48.50,47.880,48.50,4340490.0,1.012949,0.012866
3,2020-01-02 00:00:00+00:00,NVDA,Technology,237.55,240.10,236.750,240.01,1408689.0,1.014150,0.014051
4,2020-01-02 00:00:00+00:00,CGC,Healthcare,21.30,21.76,19.885,20.08,697608.0,1.094292,0.090108


### `.apply()` with `lambda` — write the function inline

Now suppose we want a Boolean flag for days when the range was unusually wide. No off-the-shelf function does this, so we define one on the spot. In Python, inline functions use the `lambda` keyword, followed by the argument and then the expression:

In [25]:
StockPrices['HighVol'] = StockPrices['RangeVolatility'].apply(lambda x: x > 1.015)
StockPrices[StockPrices['HighVol']].head()

,ts_event,symbol,sector,open,high,low,close,volume,RangeVolatility,LogVolatility,HighVol
0,2020-01-02 00:00:00+00:00,AAPL,Technology,295.25,300.93,295.080,300.85,8242956.0,1.019825,0.019631,True
1,2020-01-02 00:00:00+00:00,AMZN,Consumer Discretionary,1861.00,1900.26,1861.000,1899.50,1123936.0,1.021096,0.020877,True
4,2020-01-02 00:00:00+00:00,CGC,Healthcare,21.30,21.76,19.885,20.08,697608.0,1.094292,0.090108,True
6,2020-01-03 00:00:00+00:00,NVDA,Technology,236.50,237.80,232.110,236.08,1329086.0,1.024514,0.024219,True
7,2020-01-03 00:00:00+00:00,AMZN,Consumer Discretionary,1873.00,1886.38,1858.150,1872.80,1027769.0,1.015193,0.015078,True


**Application.** Select AAPL prices on days when the high/low ratio was at least 1.05.

In [26]:
AAPLPlus = StockPrices[(StockPrices['symbol'] == "AAPL") &
                       (StockPrices['RangeVolatility'] >= 1.05)]
AAPLPlus.head()

,ts_event,symbol,sector,open,high,low,close,volume,RangeVolatility,LogVolatility,HighVol
85,2020-01-28 00:00:00+00:00,AAPL,Technology,312.6,327.69,309.80,321.88,8867064.0,1.057747,0.056141,True
176,2020-02-24 00:00:00+00:00,AAPL,Technology,300.0,304.12,289.23,299.00,18752139.0,1.051482,0.050200,True
180,2020-02-25 00:00:00+00:00,AAPL,Technology,301.7,302.52,286.16,289.80,17637959.0,1.057171,0.055596,True
188,2020-02-26 00:00:00+00:00,AAPL,Technology,285.1,297.87,281.40,290.90,13586604.0,1.058529,0.056880,True
191,2020-02-27 00:00:00+00:00,AAPL,Technology,289.8,291.50,269.00,271.75,23220400.0,1.083643,0.080329,True


**Application.** Assume AAPL reported earnings of \$12 per share. Write a function for the P/E ratio and use it to generate a *PE_ratio* column.

In [27]:
def price_earnings(p, e):
    return p / e

earn = 12
temp = StockPrices[StockPrices['symbol'] == 'AAPL'][['symbol', 'close']].copy()
temp['PE_ratio'] = temp['close'].apply(lambda x: price_earnings(x, earn))
temp.head()

,symbol,close,PE_ratio
0,AAPL,300.85,25.070833
9,AAPL,297.79,24.815833
10,AAPL,299.01,24.917500
16,AAPL,294.51,24.542500
20,AAPL,303.58,25.298333


To remove a column, use `del` — or, more commonly, `.drop(columns=[...])`, which returns a new `DataFrame` instead of modifying in place:

In [28]:
del temp['PE_ratio']
temp.head()

,symbol,close
0,AAPL,300.85
9,AAPL,297.79
10,AAPL,299.01
16,AAPL,294.51
20,AAPL,303.58


## 8. Grouping and returns

We often want the same calculation repeated for every stock. The manual way is a loop:

In [29]:
for s in StockPrices['symbol'].unique():
    print(s, StockPrices[StockPrices['symbol'] == s]['high'].std())

AAPL 68.41701221364117
AMZN 1479.3434186432742
CSCO 5.915906710518884
NVDA 223.41700168529098
CGC 9.976732502499228


`.groupby(GROUP_ID)` does the same thing in one line, and returns a tidy object rather than printed text:

In [30]:
StockPrices.groupby("symbol")['high'].std()

symbol
AAPL      68.417012
AMZN    1479.343419
CGC        9.976733
CSCO       5.915907
NVDA     223.417002
Name: high, dtype: float64

In [31]:
StockPrices.groupby("symbol")['RangeVolatility'].median()

symbol
AAPL    1.022567
AMZN    1.026805
CGC     1.074456
CSCO    1.018788
NVDA    1.042024
Name: RangeVolatility, dtype: float64

### Application: close-to-close returns

In finance we rarely work with prices directly — we work with **returns**. The trap here is that our data stacks five stocks on top of each other, so a naive `pct_change()` would compute a "return" across the boundary between two different stocks.

`groupby(...).transform(...)` avoids this: it applies the calculation *within* each stock and returns a column aligned with the original rows.

A note on naming: we call the column `ret`, not `return`, because `return` is a reserved Python keyword and would break the regression formula in Section 10.

In [32]:
StockPrices['ret'] = StockPrices.groupby("symbol")['close'].transform(lambda x: x.pct_change(1))
StockPrices[StockPrices['symbol'] == 'AAPL'].head()

,ts_event,symbol,sector,open,high,low,close,volume,RangeVolatility,LogVolatility,HighVol,ret
0,2020-01-02 00:00:00+00:00,AAPL,Technology,295.25,300.93,295.08,300.85,8242956.0,1.019825,0.019631,True,NaN
9,2020-01-03 00:00:00+00:00,AAPL,Technology,297.20,300.58,294.31,297.79,7340919.0,1.021304,0.021080,True,-0.010171
10,2020-01-06 00:00:00+00:00,AAPL,Technology,295.53,299.96,292.75,299.01,6390771.0,1.024629,0.024330,True,0.004097
16,2020-01-07 00:00:00+00:00,AAPL,Technology,300.40,300.95,293.75,294.51,5968500.0,1.024511,0.024215,True,-0.015050
20,2020-01-08 00:00:00+00:00,AAPL,Technology,296.51,304.43,296.51,303.58,8224184.0,1.026711,0.026360,True,0.030797


## 9. Reshaping: pivots and multi-level indices

`.pivot()` reshapes long data into wide. It takes three arguments:

1. `index` — what identifies a row;
2. `columns` — what identifies a column;
3. `values` — what fills the cells.

To get a table of low prices with one row per day and one column per stock:

In [33]:
PivotLow = StockPrices.pivot(index="ts_event", columns="symbol", values="low")
PivotLow.head()

symbol,AAPL,AMZN,CGC,CSCO,NVDA
ts_event,,,,,
2020-01-02 00:00:00+00:00,295.08,1861.00,19.885,47.880,236.75
2020-01-03 00:00:00+00:00,294.31,1858.15,19.630,47.480,232.11
2020-01-06 00:00:00+00:00,292.75,1852.50,19.480,47.020,230.80
2020-01-07 00:00:00+00:00,293.75,1872.84,19.500,46.550,235.01
2020-01-08 00:00:00+00:00,296.51,1886.59,19.030,46.815,237.45


Omit `values` and `pandas` pivots *every* remaining variable, producing a two-level column index that you can then select from:

In [34]:
PivotTable = StockPrices.pivot(index="ts_event", columns="symbol")
PivotTable[['close', 'high']].head()

close                                   high  \
symbol                       AAPL     AMZN    CGC   CSCO    NVDA    AAPL   
ts_event                                                                   
2020-01-02 00:00:00+00:00  300.85  1899.50  20.08  48.50  240.01  300.93   
2020-01-03 00:00:00+00:00  297.79  1872.80  19.93  47.60  236.08  300.58   
2020-01-06 00:00:00+00:00  299.01  1899.95  20.14  47.70  237.00  299.96   
2020-01-07 00:00:00+00:00  294.51  1876.00  19.50  46.80  235.50  300.95   
2020-01-08 00:00:00+00:00  303.58  1894.75  20.30  47.79  241.03  304.43   

                                                          
symbol                        AMZN    CGC   CSCO    NVDA  
ts_event                                                  
2020-01-02 00:00:00+00:00  1900.26  21.76  48.50  240.10  
2020-01-03 00:00:00+00:00  1886.38  20.58  48.13  237.80  
2020-01-06 00:00:00+00:00  1903.33  20.25  47.85  237.37  
2020-01-07 00:00:00+00:00  1913.90  20.42  47.83  241.76  
2020-01-08 00:00:00+00:00  1910.86  20.61  47.79  242.02

### Multi-level indices

Rows can carry more than one index too. Here the natural key for an observation is the *pair* (date, stock):

In [35]:
StockPrices = StockPrices.set_index(["ts_event", "symbol"])
StockPrices.head()

sector     open     high  \
ts_event                  symbol                                             
2020-01-02 00:00:00+00:00 AAPL                Technology   295.25   300.93   
                          AMZN    Consumer Discretionary  1861.00  1900.26   
                          CSCO        Telecommunications    48.17    48.50   
                          NVDA                Technology   237.55   240.10   
                          CGC                 Healthcare    21.30    21.76   

                                       low    close     volume  \
ts_event                  symbol                                 
2020-01-02 00:00:00+00:00 AAPL     295.080   300.85  8242956.0   
                          AMZN    1861.000  1899.50  1123936.0   
                          CSCO      47.880    48.50  4340490.0   
                          NVDA     236.750   240.01  1408689.0   
                          CGC       19.885    20.08   697608.0   

                                  RangeVolatility  LogVolatility  HighVol  ret  
ts_event                  symbol                                                
2020-01-02 00:00:00+00:00 AAPL           1.019825       0.019631     True  NaN  
                          AMZN           1.021096       0.020877     True  NaN  
                          CSCO           1.012949       0.012866    False  NaN  
                          NVDA           1.014150       0.014051    False  NaN  
                          CGC            1.094292       0.090108     True  NaN

You can tell a column is part of the index because its name is **bolded** and sits slightly below the other column names.

With a multi-level index in place, `.stack()` pushes columns down into the index and `.unstack()` pulls them back up:

In [36]:
StockPricesStack = StockPrices.stack()
StockPricesStack.head(15)

ts_event                   symbol                 
2020-01-02 00:00:00+00:00  AAPL    sector                         Technology
                                   open                               295.25
                                   high                               300.93
                                   low                                295.08
                                   close                              300.85
                                   volume                          8242956.0
                                   RangeVolatility                  1.019825
                                   LogVolatility                    0.019631
                                   HighVol                              True
                                   ret                                   NaN
                           AMZN    sector             Consumer Discretionary
                                   open                               1861.0
                         

In [37]:
StockPricesStack.unstack().head()

sector    open     high  \
ts_event                  symbol                                            
2020-01-02 00:00:00+00:00 AAPL                Technology  295.25   300.93   
                          AMZN    Consumer Discretionary  1861.0  1900.26   
                          CGC                 Healthcare    21.3    21.76   
                          CSCO        Telecommunications   48.17     48.5   
                          NVDA                Technology  237.55    240.1   

                                     low   close     volume RangeVolatility  \
ts_event                  symbol                                              
2020-01-02 00:00:00+00:00 AAPL    295.08  300.85  8242956.0        1.019825   
                          AMZN    1861.0  1899.5  1123936.0        1.021096   
                          CGC     19.885   20.08   697608.0        1.094292   
                          CSCO     47.88    48.5  4340490.0        1.012949   
                          NVDA    236.75  240.01  1408689.0         1.01415   

                                 LogVolatility HighVol  ret  
ts_event                  symbol                             
2020-01-02 00:00:00+00:00 AAPL        0.019631    True  NaN  
                          AMZN        0.020877    True  NaN  
                          CGC         0.090108    True  NaN  
                          CSCO        0.012866   False  NaN  
                          NVDA        0.014051   False  NaN

## 10. OLS regression

We estimate

$$Return_{it} = \alpha + \beta \frac{High_{it}}{Low_{it}} + \varepsilon_{it},$$

to test whether a stock's return is related to its intraday price range.

This needs a new library, `statsmodels`. We use its *formula* interface, which lets us write the model in the compact `y ~ x` notation. `cov_type='HC3'` requests heteroskedasticity-robust standard errors.

In [38]:
# if statsmodels is not installed, uncomment the line below
# !pip install --upgrade statsmodels

import statsmodels.formula.api as sm

StockPrices = StockPrices.reset_index()
model = sm.ols(data=StockPrices, formula="ret ~ RangeVolatility").fit(cov_type='HC3')
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    ret   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     3.676
Date:                Mon, 17 Aug 2026   Prob (F-statistic):             0.0553
Time:                        12:30:27   Log-Likelihood:                 4872.3
No. Observations:                6585   AIC:                            -9741.
Df Residuals:                    6583   BIC:                            -9727.
Df Model:                           1                                         
Covariance Type:                  HC3                                         
===================================================================================
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -0.2095      0.109     -1.923      0.055      -0.423       0.004
RangeVolatility     0.2019      0.105      1.917      0.055      -0.005       0.408
==============================================================================
Omnibus:                    22532.697   Durbin-Watson:                   1.969
Prob(Omnibus):                  0.000   Jarque-Bera (JB):       6081719054.855
Skew:                          62.850   Prob(JB):                         0.00
Kurtosis:                    4709.370   Cond. No.                         42.6
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
"""

Every piece of the output can be pulled out individually, which is what you need when results feed into a table or a further calculation:

In [39]:
print("Coefficients:")
print(model.params)
print()
print("The range coefficient is", model.params['RangeVolatility'])

Coefficients:
Intercept         -0.209536
RangeVolatility    0.201932
dtype: float64

The range coefficient is 0.2019318191531704


In [40]:
print("p-values:")
print(model.pvalues)
print()
print("R-squared:", model.rsquared)

p-values:
Intercept          0.054506
RangeVolatility    0.055215
dtype: float64

R-squared: 0.0073430192237639025
